In [ ]:
# This creates a giant pandas DataFrame with most relevant properties like knockdown numbers, damage, angles, etc.
# NOTE: Knockdown numbers are incorrect for spikes as grounded/airborne characters tumble at differing percents.
import pandas as pd
import numpy as np
def give_props2(move_id, props, s):
    adv = hit['AdvancedOnHitProperties']

    s['Name'] = hit['Name']
    s['Damage'] = hit['Damage']
    s['KB'] = f'{hit['BaseKnockback']} + {hit['KnockbackScaling']}'
    s['KnockbackAngle'] = hit['KnockbackAngle']
    # s['SFX'] = hit['HitSoundName'].split()[-1]
    # s['VFX'] = hit['HitEffectName'].split()[-1]
    # s['Hitpause'] = adv["HitpauseMultiplier"] * hit['Damage']
    # s['Flipper'] = adv['KnockbackAngleMode'].split('::')[1]
    
    s['Reverse'] = adv['bCanReverse']
    s['Tumble'] = adv['ForceTumble']
    s['WeightIndependent'] = adv['bIgnoresWeight']
    if(adv['ForceTumble']):
        s['ZetterburnKD'] = 0
        s['ZetterburnCC'] = 0
    else:
        if (hit['KnockbackScaling'] == 0):
            if (hit['BaseKnockback'] > 26/3):
                s['ZetterburnKD'] = 0
            else:
                s['ZetterburnKD'] = 999
            if (hit['BaseKnockback'] > 26/(3 * 0.8)):
                s['ZetterburnCC'] = 0
            else:
                s['ZetterburnCC'] = 999
        else:
            localweight = 95
            if(s['WeightIndependent']):
                localweight = 100
            # s['LaReinaKD'] = np.floor(max(((26/3)-s['BaseKnockback']) * (weight + 100)/(200 * 0.12 * s['KnockbackScaling']) - s['Damage'],0))
            tumble_threshold = 26
            base_kb = hit['BaseKnockback']
            scaled_kb = hit['KnockbackScaling']
            damage = s['Damage']
            p = math.ceil((tumble_threshold / 3 - base_kb) * (localweight + 100) / (200 * 0.12 * scaled_kb) - damage)
            s['ZetterburnKD'] = max(0, p)
            cc_p = math.ceil((tumble_threshold / (3 * 0.8) - base_kb) * (localweight + 100) / (200 * 0.12 * scaled_kb) - damage)
            s['ZetterburnCC'] = max(0, cc_p)

    # If you want more columns, just add more fields like this: any further properties should be found in adv[], you can reference the following list for parameters to fetch: HitpauseMultiplier,ExtraHitpauseForOpponent,HitpauseMovementType,HitpauseMovementOffsetFromHost,HitpauseMovementStrength,SDIMultiplier,ASDIMultiplier,bCanReverse,bForceFlinch,GroundTechable,bIgnoresWeight,bAutoFloorhuggable,ProjectileInteraction,bForceKnockbackInKnockdown,bPreserveFacing,KnockbackAngleMode,HitstunMultiplier,HitfallHitstunMultiplier,ParryReaction,GrabPartnerInteraction,ExtraShieldStunInteger,ShieldDamageMultiplier,ShieldPushbackMultiplier,ShieldHitpauseMultiplier,FullChargeKnockbackMultiplier,FullChargeDamageMultiplier,FinalBaseKnockback,ForceTumble,IgnoreKnockbackArmor,PreventChaingrabsOnHit,HitstunAnimationStateOverride

    return s
        

In [ ]:
import os
import json
import math
home_path = '/home/lynnen/Dropbox/Rivals 2 Work/Export Data/Rivals2/Content/Characters' # REPLACE THIS WITH THE PATH TO YOUR EXPORTED FMODEL JSONS.
l = list()
for char in os.listdir(home_path):
    for root, dirs, files in os.walk(os.path.join(home_path, char, 'Attacks')):
        for file in files:
            move_id = file.split('_', maxsplit=2)[2].split('.')[0]
            move_data = dict()

            path = os.path.join(home_path, char, 'Attacks', file)
            if('ART_' in file):
                path = os.path.join(home_path, char, 'Attacks', 'Articles', file)

            with open(path) as f:
                lexicon = dict()
                props = json.load(f)[0]
                if('Properties' not in props.keys()):
                    continue
                props = props['Properties']
                list_of_boxes = list()
                if('HitboxAttributes' in props.keys()):
                    for hit in props['HitboxAttributes']:
                        list_of_boxes.append(hit['OnHitPropertiesName'])
                if('HitboxOnHitProperties' in props.keys()):
                    for hit in props['HitboxOnHitProperties']:
                        if(hit['Name'] not in list_of_boxes):
                            continue
                        s = dict()
                        s['chara'] = char
                        s['move'] = move_id
                        s = give_props2(move_id, props, s)
                        l.append(s)
p = pd.DataFrame(l)
pd.set_option('display.max_rows', None)
p.keys()
p[p.chara == 'Orcane']

In [1]:
# Examining windows of a given character and file
import json
import pandas as pd
chara = 'LaReina'
attack = 'Dthrow'
f = open(f'/home/lynnen/Dropbox/Rivals 2 Work/Export Data/Rivals2/Content/Characters/{chara}/Attacks/ATT_{chara[0:3].title()}_{attack}.json')
props = json.load(f)[0]
pd.DataFrame(props['Properties']['Windows'])

,StringTableKey,WindowLengthFrames,IasaFrame,AnimationStartFrame,AnimationLengthFrames,NextWindowStringTableKey,WindowCancels,VelocityData,MovementProperties,HitboxWindows,bRefreshHasHit,HurtboxStateChanges,AdvancedProperties,LedgeProperties
0,Startup,22,-1,0,22,Active,[],[],"{'Gravity': -1.0, 'MaxFallSpeed': -1.0, 'Frict...",[],True,[],"{'WindowArmor': -1, 'AerialAutocancelFrame': -...","{'CanGrabLedgeOnFrame': -1, 'LedgeGrabBoxOffse..."
1,Active,1,-1,22,1,Recovery,[],[],"{'Gravity': -1.0, 'MaxFallSpeed': -1.0, 'Frict...","[{'HitboxName': 'Slam', 'StartFrame': 0, 'Leng...",True,[],"{'WindowArmor': -1, 'AerialAutocancelFrame': -...","{'CanGrabLedgeOnFrame': -1, 'LedgeGrabBoxOffse..."
2,Recovery,28,24,23,20,,[],[],"{'Gravity': -1.0, 'MaxFallSpeed': -1.0, 'Frict...",[],True,[],"{'WindowArmor': -1, 'AerialAutocancelFrame': -...","{'CanGrabLedgeOnFrame': -1, 'LedgeGrabBoxOffse..."
